In [72]:
!which python

/Users/apabook/Desktop/ds-deployment/venv/bin/python


In [73]:
import json
from urllib.error import HTTPError

import pandas as pd
import requests

# Complete dataspace flow

## Consts


In [74]:
data_space_provider = "http://127.0.0.1:1200"
data_space_consumer = "http://127.0.0.1:1100"
api_push_endpoint = "http://localhost:8082/subscriptions/subscribe"
api_push_description = {
    "url": "http://host.docker.internal:1200",
    "event_type": "hotel_water_usage"
}

data_space_provider_participant_id = ""
data_space_provider_participant_token = ""
data_space_consumer_participant_id = ""
data_space_consumer_participant_token = ""
catalog_id = ""
dataset_id = ""
distribution_id = ""
data_service_id = ""
policy_id = ""
target_id = ""

cn_consumer_id = ""
cn_provider_id = ""
agreement_id = ""
transfer_consumer_id = ""
transfer_provider_id = ""
consumer_data_address = ""

json_header = {
    "Content-Type": "application/json",
}

## Wallets setup

## Participants onboarding

### Current Participants information

In [75]:
url = data_space_provider + "/api/v1/mates/myself"
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "participant_id": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiIwLW5uempxdDNQVXZtMUVPaEVxNUNxQ05WcEVUM3I4S0Y3VWRKSDJlRFV1dDNhSFlFWGFxbG51SXZ4TUVmN0J1dkhMVnlUaW95U1BoMV9TaDF0ZnBwdF9aNXpXU1ZZUGp0ekZ0QlJsQ0pQYzNfc2YtelRIQURMMmNyYW84WmxXa1FFR2ZtX1g3Y2QtMkd5bmVuR0k0a01rLU9vRERUWEJNRmVoOVdtR1dyMGZBeHJ4Qkp3Unh4QTExODNIOWo1QkY3dlpVZ0Y3akYzV28xcVlva0Z4emJ0Q2FhWWkzTmE3aFYtWWRIYkg3QUhkYnlJa1JvTHhhVG9tb3ktRHRmb0dSSGRzTmpzNDFqZUwxTG5XczJENnVJMm43QTBFQ1hIaHJjZTlMbkt6eXNMWktRM3Ffa2hxR0UzRTZJT1djajRMT1NvNGJ4SXktZzNVT0NpeElSelJ2bFEifQ",
    "participant_slug": "Myself",
    "participant_type": "Provider",
    "base_url": "http://127.0.0.1:1200",
    "token": null,
    "saved_at": "2025-12-01T11:33:32.729305",
    "last_interaction": "2025-12-01T11:33:32.729330",
    "is_me": true
}


In [76]:
url = data_space_consumer + "/api/v1/mates/myself"
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()
print(json.dumps(response_as_json, indent=4))

{
    "participant_id": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiJ4T0F3WlpkQ3AwQ0dCQV9tMVVHR1VXcEM5THJlcmJjbDMxblJoZThzNzd6R204aXA3YUgtUW5NNUNLbEo1YXlvUDhKTV9rLWhVQ0V2Q09qNWw5VmNhN0RfNTdRMjRJUEtHN0dQSGdnYzZ5bGxwZFlnSkVlY2xUcXJGcjdGUFBxcThDQzY1WVpfS1hGY3ByOHJGRVFEUGtkc3gzbU1qTkYzQUpNNzBpaEItdGtOOWxxcHZLb0h6MzJRaHRybVhFY2t1X1NtVFlFTU9GVU0xd0I1TE4xNmN4ZzIxZXYwMnZGdlVCYmdWZENTNHZocjNHNlNMM2dPZXVzd3BZaTZtTTZ6clloY3lkaDJaRjZCUEZVMS1NNk9tcENGRnd6SFVWNTdqMHJMLUR0OU1Ja1FUWmxXam5kcVVOZG1Ecnlyd0E1OU8zS1lWYWY5cmUwdW9QS3ZGUmRmY3cifQ",
    "participant_slug": "Myself",
    "participant_type": "Consumer",
    "base_url": "http://127.0.0.1:1100",
    "token": null,
    "saved_at": "2025-12-01T11:33:29.398280",
    "last_interaction": "2025-12-01T11:33:29.398505",
    "is_me": true
}


### Provider and Consumer ids

In [77]:
url = data_space_provider + "/api/v1/mates/all"
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()

for participant in response_as_json:
    if participant.get("participant_type") == "Consumer":
        data_space_consumer_participant_id = participant["participant_id"]
        data_space_consumer_participant_token = participant["token"]
        break

print(json.dumps(response_as_json, indent=4))

[
    {
        "participant_id": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiIwLW5uempxdDNQVXZtMUVPaEVxNUNxQ05WcEVUM3I4S0Y3VWRKSDJlRFV1dDNhSFlFWGFxbG51SXZ4TUVmN0J1dkhMVnlUaW95U1BoMV9TaDF0ZnBwdF9aNXpXU1ZZUGp0ekZ0QlJsQ0pQYzNfc2YtelRIQURMMmNyYW84WmxXa1FFR2ZtX1g3Y2QtMkd5bmVuR0k0a01rLU9vRERUWEJNRmVoOVdtR1dyMGZBeHJ4Qkp3Unh4QTExODNIOWo1QkY3dlpVZ0Y3akYzV28xcVlva0Z4emJ0Q2FhWWkzTmE3aFYtWWRIYkg3QUhkYnlJa1JvTHhhVG9tb3ktRHRmb0dSSGRzTmpzNDFqZUwxTG5XczJENnVJMm43QTBFQ1hIaHJjZTlMbkt6eXNMWktRM3Ffa2hxR0UzRTZJT1djajRMT1NvNGJ4SXktZzNVT0NpeElSelJ2bFEifQ",
        "participant_slug": "Myself",
        "participant_type": "Provider",
        "base_url": "http://127.0.0.1:1200",
        "token": null,
        "saved_at": "2025-12-01T11:33:32.729305",
        "last_interaction": "2025-12-01T11:33:32.729330",
        "is_me": true
    },
    {
        "participant_id": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiJ4T0F3WlpkQ3AwQ0dCQV9tMVVHR1VXcEM5THJlcmJjbDMxblJoZThzNzd6R204aXA3YUgtUW5NNUNLbEo1YXlvUDhKTV9r

In [78]:
url = data_space_consumer + "/api/v1/mates/all"
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()

for participant in response_as_json:
    if participant.get("participant_type") == "Provider":
        data_space_provider_participant_id = participant["participant_id"]
        data_space_provider_participant_token = participant["token"]
        break

print(json.dumps(response_as_json, indent=4))

[
    {
        "participant_id": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiJ4T0F3WlpkQ3AwQ0dCQV9tMVVHR1VXcEM5THJlcmJjbDMxblJoZThzNzd6R204aXA3YUgtUW5NNUNLbEo1YXlvUDhKTV9rLWhVQ0V2Q09qNWw5VmNhN0RfNTdRMjRJUEtHN0dQSGdnYzZ5bGxwZFlnSkVlY2xUcXJGcjdGUFBxcThDQzY1WVpfS1hGY3ByOHJGRVFEUGtkc3gzbU1qTkYzQUpNNzBpaEItdGtOOWxxcHZLb0h6MzJRaHRybVhFY2t1X1NtVFlFTU9GVU0xd0I1TE4xNmN4ZzIxZXYwMnZGdlVCYmdWZENTNHZocjNHNlNMM2dPZXVzd3BZaTZtTTZ6clloY3lkaDJaRjZCUEZVMS1NNk9tcENGRnd6SFVWNTdqMHJMLUR0OU1Ja1FUWmxXam5kcVVOZG1Ecnlyd0E1OU8zS1lWYWY5cmUwdW9QS3ZGUmRmY3cifQ",
        "participant_slug": "Myself",
        "participant_type": "Consumer",
        "base_url": "http://127.0.0.1:1100",
        "token": null,
        "saved_at": "2025-12-01T11:33:29.398280",
        "last_interaction": "2025-12-01T11:33:29.398505",
        "is_me": true
    },
    {
        "participant_id": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiJtcHBQUnNOOURBSmZDNDNnaWNPSUZLTzBsT3l5d3lZSHZfcGZYWU4xNUhsRGVXOXZPZU9UVjBzTWdYUjZNMkJLc3pncG1a

## Create Catalog instances and metadata

In [79]:
url = data_space_provider + "/api/v1/catalogs"
payload = {
    "foaf:homepage": "My dataspace catalog",
    "dct:title": "My dataspace catalog",

}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
catalog_id = response_as_json["@id"]
print(json.dumps(response_as_json, indent=4))

{
    "@context": [
        "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "Catalog",
    "@id": "urn:uuid:8f10846c-aebd-4425-85ba-300f43a2db3f",
    "homepage": "My dataspace catalog",
    "theme": "",
    "keyword": "",
    "conformsTo": null,
    "creator": null,
    "identifier": "urn:uuid:8f10846c-aebd-4425-85ba-300f43a2db3f",
    "issued": "2025-12-03T09:30:34.439695",
    "modified": null,
    "title": "My dataspace catalog",
    "description": [],
    "participantId": "urn:uuid:12dbf581-1dda-4a01-ad65-b61240ed8f6a",
    "hasPolicy": [],
    "extraFields": null,
    "catalog": [],
    "dataset": [],
    "service": []
}


In [80]:
url = data_space_provider + "/api/v1/catalogs/{}/datasets".format(catalog_id)
payload = {
    "dct:title": "My dataspace dataset",
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
dataset_id = response_as_json["@id"]
print(json.dumps(response_as_json, indent=4))

{
    "@context": [
        "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "Dataset",
    "@id": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97",
    "theme": "",
    "keyword": "",
    "conformsTo": null,
    "creator": null,
    "identifier": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97",
    "issued": "2025-12-03T09:30:34.891692",
    "modified": null,
    "title": "My dataspace dataset",
    "description": [],
    "hasPolicy": [],
    "extraFields": null,
    "distribution": []
}


In [81]:
url = data_space_provider + "/api/v1/catalogs/{}/data-services".format(catalog_id)
payload = {
    "dcat:endpointURL": api_push_endpoint,
    "dcat:endpointDescription": str(api_push_description)
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
data_service_id = response_as_json["@id"]
print(json.dumps(response_as_json, indent=4))

{
    "@context": [
        "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "DataService",
    "@id": "urn:uuid:b03ebaaf-c7bc-41de-bd52-64b806268b8f",
    "theme": "",
    "keyword": "",
    "endpointDescription": "{'url': 'http://host.docker.internal:1200', 'event_type': 'hotel_water_usage'}",
    "endpointURL": "http://localhost:8082/subscriptions/subscribe",
    "conformsTo": null,
    "creator": null,
    "identifier": "urn:uuid:b03ebaaf-c7bc-41de-bd52-64b806268b8f",
    "issued": "2025-12-03T09:30:35.282362",
    "modified": null,
    "title": null,
    "description": [],
    "hasPolicy": [],
    "extraFields": null
}


In [82]:
url = data_space_provider + "/api/v1/catalogs/{}/datasets/{}/distributions".format(catalog_id, dataset_id)
payload = {
    "dcat:accessService": data_service_id,
    "dct:formats": "http+push"
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
distribution_id = response_as_json["@id"]
print(json.dumps(response_as_json, indent=4))

{
    "@context": [
        "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "Distribution",
    "@id": "urn:uuid:47456167-7a95-4b18-9ea9-f7bd57d8ae5a",
    "accessService": {
        "@context": [
            "https://w3id.org/dspace/2025/1/context.jsonld"
        ],
        "@type": "DataService",
        "@id": "urn:uuid:b03ebaaf-c7bc-41de-bd52-64b806268b8f",
        "theme": "",
        "keyword": "",
        "endpointDescription": "{'url': 'http://host.docker.internal:1200', 'event_type': 'hotel_water_usage'}",
        "endpointURL": "http://localhost:8082/subscriptions/subscribe",
        "conformsTo": null,
        "creator": null,
        "identifier": "urn:uuid:b03ebaaf-c7bc-41de-bd52-64b806268b8f",
        "issued": "2025-12-03T09:30:35.282362",
        "modified": null,
        "title": null,
        "description": [],
        "hasPolicy": [],
        "extraFields": null
    },
    "identifier": "urn:uuid:47456167-7a95-4b18-9ea9-f7bd57d8ae5a",
    "issued"

## Create policies

In [83]:
url = data_space_provider + "/api/v1/datasets/{}/policies".format(dataset_id)
payload = {
    "permission": [
        {
            "action": "use",
            "constraint": [
                {
                    "rightOperand": "user",
                    "leftOperand": "did:web:hola.es",
                    "operator": "eq"
                }
            ]
        }

    ],
    "obligation": [],
    "prohibition": []
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
policy_id = response_as_json["@id"]
target_id = response_as_json["target"]
print(json.dumps(response_as_json, indent=4))

{
    "@id": "urn:uuid:34b4fed2-bbdd-4751-afe3-438d48197c02",
    "permission": [
        {
            "action": "use",
            "constraint": [
                {
                    "rightOperand": "user",
                    "leftOperand": "did:web:hola.es",
                    "operator": "eq"
                }
            ]
        }
    ],
    "obligation": [],
    "@type": "Offer",
    "prohibition": [],
    "target": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97"
}


## Contract Negotiation

In [84]:
url = data_space_consumer + "/api/v1/negotiations/rpc/setup-request"
payload = {
    "providerParticipantId": data_space_provider_participant_id,
    "offer": {
        "@id": policy_id,  # policy id
        "target": target_id,  # data id
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
cn_consumer_id = response_as_json["consumerPid"]
cn_provider_id = response_as_json["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "offer": {
    "@id": "urn:uuid:34b4fed2-bbdd-4751-afe3-438d48197c02",
    "permission": [
      {
        "action": "use"
      }
    ],
    "@type": "Offer",
    "target": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97"
  },
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
    "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
    "state": "REQUESTED"
  }
}


In [85]:
url = data_space_provider + "/api/v1/negotiations/rpc/setup-offer"
payload = {
    "consumerParticipantId": data_space_consumer_participant_id,
    "consumerPid": cn_consumer_id,  # remove to test offer from provider
    "providerPid": cn_provider_id,  # remove to test offer from provider
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "consumerParticipantId": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiJ4T0F3WlpkQ3AwQ0dCQV9tMVVHR1VXcEM5THJlcmJjbDMxblJoZThzNzd6R204aXA3YUgtUW5NNUNLbEo1YXlvUDhKTV9rLWhVQ0V2Q09qNWw5VmNhN0RfNTdRMjRJUEtHN0dQSGdnYzZ5bGxwZFlnSkVlY2xUcXJGcjdGUFBxcThDQzY1WVpfS1hGY3ByOHJGRVFEUGtkc3gzbU1qTkYzQUpNNzBpaEItdGtOOWxxcHZLb0h6MzJRaHRybVhFY2t1X1NtVFlFTU9GVU0xd0I1TE4xNmN4ZzIxZXYwMnZGdlVCYmdWZENTNHZocjNHNlNMM2dPZXVzd3BZaTZtTTZ6clloY3lkaDJaRjZCUEZVMS1NNk9tcENGRnd6SFVWNTdqMHJMLUR0OU1Ja1FUWmxXam5kcVVOZG1Ecnlyd0E1OU8zS1lWYWY5cmUwdW9QS3ZGUmRmY3cifQ",
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "offer": {
    "@id": "urn:uuid:34b4fed2-bbdd-4751-afe3-438d48197c02",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "@type": "Offer",
    "target": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97"
  },
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    

In [86]:
url = data_space_consumer + "/api/v1/negotiations/rpc/setup-request"
payload = {
    "providerParticipantId": data_space_provider_participant_id,
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,  # policy id
        "target": target_id,  # data id
        "@type": "Offer",
        "permission": [
            {
                "action": "use"

            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
cn_consumer_id = response_as_json["consumerPid"]
cn_provider_id = response_as_json["providerPid"]
print(json.dumps(response_as_json, indent=2))

{
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "offer": {
    "@id": "urn:uuid:34b4fed2-bbdd-4751-afe3-438d48197c02",
    "permission": [
      {
        "action": "use"
      }
    ],
    "@type": "Offer",
    "target": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97"
  },
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
    "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
    "state": "REQUESTED"
  }
}


In [87]:
url = data_space_provider + "/api/v1/negotiations/rpc/setup-offer"
payload = {
    "consumerParticipantId": data_space_consumer_participant_id,
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
    "offer": {
        "@id": policy_id,
        "target": target_id,
        "@type": "Offer",
        "permission": [
            {
                "action": "supermegause"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "consumerParticipantId": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiJ4T0F3WlpkQ3AwQ0dCQV9tMVVHR1VXcEM5THJlcmJjbDMxblJoZThzNzd6R204aXA3YUgtUW5NNUNLbEo1YXlvUDhKTV9rLWhVQ0V2Q09qNWw5VmNhN0RfNTdRMjRJUEtHN0dQSGdnYzZ5bGxwZFlnSkVlY2xUcXJGcjdGUFBxcThDQzY1WVpfS1hGY3ByOHJGRVFEUGtkc3gzbU1qTkYzQUpNNzBpaEItdGtOOWxxcHZLb0h6MzJRaHRybVhFY2t1X1NtVFlFTU9GVU0xd0I1TE4xNmN4ZzIxZXYwMnZGdlVCYmdWZENTNHZocjNHNlNMM2dPZXVzd3BZaTZtTTZ6clloY3lkaDJaRjZCUEZVMS1NNk9tcENGRnd6SFVWNTdqMHJMLUR0OU1Ja1FUWmxXam5kcVVOZG1Ecnlyd0E1OU8zS1lWYWY5cmUwdW9QS3ZGUmRmY3cifQ",
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "offer": {
    "@id": "urn:uuid:34b4fed2-bbdd-4751-afe3-438d48197c02",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "@type": "Offer",
    "target": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97"
  },
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    

In [88]:
url = data_space_consumer + "/api/v1/negotiations/rpc/setup-acceptance"
payload = {
    "providerParticipantId": data_space_provider_participant_id,
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
    "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
    "state": "ACCEPTED"
  }
}


In [89]:
url = data_space_provider + "/api/v1/negotiations/rpc/setup-agreement"
payload = {
    "consumerParticipantId": data_space_consumer_participant_id,
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
agreement_id = response_as_json["agreement"]["@id"]
print(json.dumps(response_as_json, indent=2))

{
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "agreement": {
    "@id": "urn:uuid:f12ad91d-3893-4f44-be3a-b410fa6e8ec9",
    "permission": [
      {
        "action": "supermegause"
      }
    ],
    "@type": "Agreement",
    "target": "urn:uuid:20b48f80-0431-4d38-8202-7c6b83b43c97",
    "assigner": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiIwLW5uempxdDNQVXZtMUVPaEVxNUNxQ05WcEVUM3I4S0Y3VWRKSDJlRFV1dDNhSFlFWGFxbG51SXZ4TUVmN0J1dkhMVnlUaW95U1BoMV9TaDF0ZnBwdF9aNXpXU1ZZUGp0ekZ0QlJsQ0pQYzNfc2YtelRIQURMMmNyYW84WmxXa1FFR2ZtX1g3Y2QtMkd5bmVuR0k0a01rLU9vRERUWEJNRmVoOVdtR1dyMGZBeHJ4Qkp3Unh4QTExODNIOWo1QkY3dlpVZ0Y3akYzV28xcVlva0Z4emJ0Q2FhWWkzTmE3aFYtWWRIYkg3QUhkYnlJa1JvTHhhVG9tb3ktRHRmb0dSSGRzTmpzNDFqZUwxTG5XczJENnVJMm43QTBFQ1hIaHJjZTlMbkt6eXNMWktRM3Ffa2hxR0UzRTZJT1djajRMT1NvNGJ4SXktZzNVT0NpeElSelJ2bFEifQ",
    "assignee": "did:jwk:eyJrdHkiOiJSU0EiLCJlIjoiQVFBQiIsIm4iOiJ4T0F3WlpkQ3AwQ0dCQV9tMVVHR1VXcEM5T

In [90]:
url = data_space_consumer + "/api/v1/negotiations/rpc/setup-verification"
payload = {
    "providerParticipantId": data_space_provider_participant_id,
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
    "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
    "state": "VERIFIED"
  }
}


In [91]:
url = data_space_provider + "/api/v1/negotiations/rpc/setup-finalization"
payload = {
    "consumerParticipantId": data_space_consumer_participant_id,
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
  "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "ContractNegotiation",
    "providerPid": "urn:uuid:10ec7d72-d02d-4263-883a-43bb26f40756",
    "consumerPid": "urn:uuid:672019f7-f49e-4387-88f8-bbf1dd6953c9",
    "state": "FINALIZED"
  }
}


## Transfer Negotiation


In [92]:
url = data_space_consumer + "/api/v1/transfers/rpc/setup-request"
payload = {
    "providerParticipantId": data_space_provider_participant_id,
    "agreementId": agreement_id,
    "format": "http+push",
    "dataAddress": {
        "@type": "DataAddress",
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {
                "@type": "EndpointProperty",
                "name": "authorization",
                "value": "TOKEN-ABCDEFG"
            },
            {
                "@type": "EndpointProperty",
                "name": "authType",
                "value": "bearer"
            }
        ]
    }
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_consumer_id = response_as_json["consumerPid"]
transfer_provider_id = response_as_json["providerPid"]
callback_address = response_as_json["callbackAddress"]
print(json.dumps(response_as_json, indent=2))

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [26]:
url = data_space_provider + "/api/v1/transfers/rpc/setup-start"
payload = {
    "consumerParticipantId": data_space_consumer_participant_id,
    "consumerCallbackAddress": callback_address,
    "providerPid": transfer_provider_id,
    "consumerPid": transfer_consumer_id
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()

print(json.dumps(response_as_json, indent=2))

{
  "providerPid": "urn:uuid:9db69bb1-723d-4229-81e3-bc4d9fd96bfd",
  "consumerPid": "urn:uuid:e591e387-82b9-4489-8aab-37558d8f1d50",
  "dataAddress": {
    "@type": "DataAddress",
    "endpointType": "http",
    "endpoint": "http://host.docker.internal:1200/data/urn:uuid:9db69bb1-723d-4229-81e3-bc4d9fd96bfd",
    "endpointProperties": [
      {
        "@type": "EndpointProperty",
        "name": "authType",
        "value": ""
      },
      {
        "@type": "EndpointProperty",
        "name": "authorization",
        "value": ""
      }
    ]
  },
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "providerPid": "urn:uuid:9db69bb1-723d-4229-81e3-bc4d9fd96bfd",
    "consumerPid": "urn:uuid:e591e387-82b9-4489-8aab-37558d8f1d50",
    "state": "STARTED"
  }
}


In [27]:
url = data_space_consumer + "/api/v1/dataplane/" + transfer_consumer_id
response = requests.request("GET", url, headers=json_header)
response_as_json = response.json()
consumer_data_address = response_as_json["process_address"]["url"]
print(json.dumps(response_as_json, indent=2))

{
  "id": "urn:uuid:e591e387-82b9-4489-8aab-37558d8f1d50",
  "process_direction": "PULL",
  "upstream_hop": {
    "protocol": "",
    "url": "",
    "auth_type": "",
    "auth_content": ""
  },
  "downstream_hop": {
    "protocol": "",
    "url": "",
    "auth_type": "",
    "auth_content": ""
  },
  "process_address": {
    "protocol": "http",
    "url": "http://host.docker.internal:1100/data/urn:uuid:e591e387-82b9-4489-8aab-37558d8f1d50",
    "auth_type": "",
    "auth_content": ""
  },
  "created_at": "2025-10-23T13:11:59.470263",
  "updated_at": "2025-10-23T13:11:59.541849",
  "state": "STARTED"
}


In [28]:
try:
    df = pd.read_json(consumer_data_address)
    print(df)
except HTTPError as e:
    print(e)


URLError: <urlopen error [Errno 8] nodename nor servname provided, or not known>

In [29]:
url = data_space_consumer + "/api/v1/transfers/rpc/setup-suspension"
payload = {
    "providerParticipantId": data_space_provider_participant_id,
    "providerPid": transfer_provider_id,
    "consumerPid": transfer_consumer_id,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "providerPid": "urn:uuid:b44cc898-76d4-4831-bd34-66e7e5f05c66",
  "consumerPid": "urn:uuid:6e06df6f-a1ee-4a12-9de8-a1264524c1ad",
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "providerPid": "urn:uuid:b44cc898-76d4-4831-bd34-66e7e5f05c66",
    "consumerPid": "urn:uuid:6e06df6f-a1ee-4a12-9de8-a1264524c1ad",
    "state": "SUSPENDED"
  }
}


In [30]:
try:
    df = pd.read_json(consumer_data_address)
    print(df)
except HTTPError as e:
    print(e)

HTTP Error 403: Forbidden


In [31]:
url = data_space_consumer + "/api/v1/transfers/rpc/setup-start"
payload = {
    "providerParticipantId": data_space_provider_participant_id,
    "providerPid": transfer_provider_id,
    "consumerPid": transfer_consumer_id,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "providerPid": "urn:uuid:b44cc898-76d4-4831-bd34-66e7e5f05c66",
  "consumerPid": "urn:uuid:6e06df6f-a1ee-4a12-9de8-a1264524c1ad",
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "providerPid": "urn:uuid:b44cc898-76d4-4831-bd34-66e7e5f05c66",
    "consumerPid": "urn:uuid:6e06df6f-a1ee-4a12-9de8-a1264524c1ad",
    "state": "STARTED"
  }
}


In [32]:
try:
    df = pd.read_json(consumer_data_address)
    print(df)
except HTTPError as e:
    print(e)

     postId   id                                               name  \
0         1    1                       id labore ex et quam laborum   
1         1    2          quo vero reiciendis velit similique earum   
2         1    3                      odio adipisci rerum aut animi   
3         1    4                                     alias odio sit   
4         1    5              vero eaque aliquid doloribus et culpa   
..      ...  ...                                                ...   
495     100  496  et occaecati asperiores quas voluptas ipsam no...   
496     100  497             doloribus dolores ut dolores occaecati   
497     100  498                           dolores minus aut libero   
498     100  499  excepturi sunt cum a et rerum quo voluptatibus...   
499     100  500                                 ex eaque eum natus   

                            email  \
0              Eliseo@gardner.biz   
1          Jayne_Kuhic@sydney.com   
2             Nikita@garfield.biz   

In [33]:
url = data_space_provider + "/api/v1/transfers/rpc/setup-suspension"
payload = {
    "consumerParticipantId": data_space_consumer_participant_id,
    "consumerCallbackAddress": callback_address,
    "providerPid": transfer_provider_id,
    "consumerPid": transfer_consumer_id,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "providerPid": "urn:uuid:b44cc898-76d4-4831-bd34-66e7e5f05c66",
  "consumerPid": "urn:uuid:6e06df6f-a1ee-4a12-9de8-a1264524c1ad",
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "providerPid": "urn:uuid:b44cc898-76d4-4831-bd34-66e7e5f05c66",
    "consumerPid": "urn:uuid:6e06df6f-a1ee-4a12-9de8-a1264524c1ad",
    "state": "SUSPENDED"
  }
}


In [29]:
url = data_space_provider + "/api/v1/transfers/rpc/setup-completion"
payload = {
    "consumerParticipantId": data_space_consumer_participant_id,
    "consumerCallbackAddress": callback_address,
    "providerPid": transfer_provider_id,
    "consumerPid": transfer_consumer_id,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))

{
  "providerPid": "urn:uuid:9db69bb1-723d-4229-81e3-bc4d9fd96bfd",
  "consumerPid": "urn:uuid:e591e387-82b9-4489-8aab-37558d8f1d50",
  "message": {
    "@context": [
      "https://w3id.org/dspace/2025/1/context.jsonld"
    ],
    "@type": "TransferProcess",
    "providerPid": "urn:uuid:9db69bb1-723d-4229-81e3-bc4d9fd96bfd",
    "consumerPid": "urn:uuid:e591e387-82b9-4489-8aab-37558d8f1d50",
    "state": "COMPLETED"
  }
}
